In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
import wandb

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [17]:
# Normal dataset
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Augmented dataset
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [18]:
from torch.utils.data import DataLoader, random_split

# Choose transform_basic or transform_aug
dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                       download=True, transform=transform_basic)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)

In [19]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, embed_dim=64):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(in_ch, embed_dim,
                             kernel_size=patch_size,
                             stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

In [20]:
class ViT(nn.Module):
    def __init__(self, embed_dim=64, num_heads=4, depth=4, num_classes=10):
        super().__init__()

        self.patch_embed = PatchEmbedding()
        num_patches = self.patch_embed.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        B, N, E = x.shape

        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed
        x = self.encoder(x)

        cls_out = x[:, 0]
        return self.fc(cls_out)

In [21]:
def get_resnet():
    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

In [22]:
ce_loss = nn.CrossEntropyLoss()

ls_loss = nn.CrossEntropyLoss(label_smoothing=0.1)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = nn.CrossEntropyLoss()(inputs, targets)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce)

In [23]:
def get_optimizer(name, model):
    if name == "SGD":
        return optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    elif name == "Adam":
        return optim.Adam(model.parameters(), lr=0.001)
    elif name == "RMSprop":
        return optim.RMSprop(model.parameters(), lr=0.001)

In [24]:
def train(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [25]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = loss_fn(outputs, y)

            total_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    return total_loss / len(loader), acc

In [26]:
model = ViT()   # OR get_resnet()
model = model.to(device)

optimizer = get_optimizer("Adam", model)
loss_fn = ce_loss

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [27]:
wandb.init(project="ViT-vs-ResNet")

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    train_loss = train(model, train_loader, optimizer, loss_fn)
    val_loss, acc = evaluate(model, val_loader, loss_fn)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Acc={acc:.4f}")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc
    })

Epoch 1: Train Loss=2.3204, Val Loss=2.3193, Acc=0.1000


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)

print("Test Accuracy:", test_acc)